# LeRobot Ray Data Datasource — Example Usage

This notebook downloads the `lerobot/berkeley_autolab_ur5` dataset and reads it
using `LeRobotDatasource` with two different parallelism modes.

## 1. Download the dataset

`LeRobotDataset` pulls the dataset from the Hugging Face Hub on first use and
caches it locally. We grab the `root` path for the datasource to read from.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset("lerobot/berkeley_autolab_ur5", video_backend="pyav")
root = str(ds.root)
print(f"Dataset root: {root}")
print(f"Episodes: {ds.meta.total_episodes}, Frames: {ds.meta.total_frames}")

## 2. Read with the default parallelism mode (`file_group`)

`file_group` groups episodes that share the same video files into a single read
task, so each mp4 is opened only once per task. This is the recommended default.

In [ ]:
import ray
from lerobot_datasource import LeRobotDatasource

ray.init(ignore_reinit_error=True)

ds = ray.data.read_datasource(LeRobotDatasource(root=root))
print(ds.schema())

In [ ]:
sample = ds.take(5)
for row in sample:
    print(
        f"ep={row['episode_index']:>3d}  frame={row['frame_index']:>4d}  "
        f"task={row['task']!r:.60s}"
    )

## 3. Read with `episode` parallelism mode

`episode` mode creates one read task per episode — maximum parallelism, but each
video file may be opened by multiple tasks. Useful for small local datasets
where I/O cost is low.

In [ ]:
from lerobot_datasource import ParallelismMode

ds_episode = ray.data.read_datasource(
    LeRobotDatasource(root=root, parallelism_mode=ParallelismMode.EPISODE)
)
print(ds_episode)

In [ ]:
sample = ds_episode.take(5)
for row in sample:
    print(
        f"ep={row['episode_index']:>3d}  frame={row['frame_index']:>4d}  "
        f"task={row['task']!r:.60s}"
    )

In [ ]:
ray.shutdown()